# Upload Models to HuggingFace Hub

Upload trained brain segmentation model **weights** to HuggingFace.
Runs on **Databricks** (reads from DBFS) or **locally** (reads from `models/`).

**No GPU needed** — this is just file transfer.

| What | Notebook |
|------|----------|
| Upload model weights | **This notebook** |
| Upload/update papers + READMEs | `upload_papers_to_hf.ipynb` (run locally) |

## Usage
1. Run all cells in order
2. Change `UPLOAD_SPECIES` in Cell 2 to control which model(s) to upload

In [1]:
# Cell 0 — Install dependencies (Databricks only)
# Uncomment these two lines if running on Databricks:
# %pip install huggingface_hub python-dotenv
# dbutils.library.restartPython()

In [2]:
# Cell 1 — Detect environment and check model availability

import os
from pathlib import Path
from dotenv import load_dotenv

# --- Load .env file (looks in repo root) ---
for _env_candidate in [Path(".env"), Path("../.env")]:
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        print(f"Loaded .env from {_env_candidate.resolve()}")
        break

# --- Auto-detect models directory ---
_model_candidates = [
    Path("/dbfs/FileStore/allen_brain_data/models"),
    Path("models"),
    Path("../models"),
]
MODELS_DIR = next((p for p in _model_candidates if p.exists()), Path("models"))
_is_dbfs = "dbfs" in str(MODELS_DIR).lower()

print(f"Models dir: {MODELS_DIR}  ({'DBFS' if _is_dbfs else 'local'})")

# DBFS uses different directory names than local
MODEL_CONFIGS = {
    "mouse": {
        "local_dir": MODELS_DIR / ("final-200ep" if _is_dbfs else "dinov2-upernet-final"),
        "repo_suffix": "mouse",
    },
    "human": {
        "local_dir": MODELS_DIR / ("human-allen-depth3" if _is_dbfs else "human-depth3"),
        "repo_suffix": "human",
    },
    "human-bigbrain": {
        "local_dir": MODELS_DIR / "human-bigbrain",
        "repo_suffix": "human-bigbrain",
    },
}

HF_USERNAME = "Noel-Niko"
MODEL_BASE = "dinov2-upernet-20260322-histology-annotation"
REQUIRED_FILES = ["config.json", "preprocessor_config.json"]
WEIGHT_FILES = ["model.safetensors", "pytorch_model.bin"]

HF_TOKEN = os.environ.get("HUGGING_FACE_TOKEN")
print(f"HF token:   {'found (' + HF_TOKEN[:8] + '...)' if HF_TOKEN else 'NOT SET'}")

print("\n" + "=" * 55)
for name, cfg in MODEL_CONFIGS.items():
    d = cfg["local_dir"]
    exists = d.exists()
    ok = (
        exists
        and all((d / f).exists() for f in REQUIRED_FILES)
        and any((d / f).exists() for f in WEIGHT_FILES)
    )
    icon = "OK" if ok else "!!"
    print(f"  [{icon}] {name:16s}  {d}")
print("=" * 55)

Loaded .env from /Users/xnxn040/PycharmProjects/Personal-Projects/histological-image-analysis/.env
Models dir: ../models  (local)
HF token:   found (hf_FnOaZ...)

  [OK] mouse             ../models/dinov2-upernet-final
  [OK] human             ../models/human-depth3
  [OK] human-bigbrain    ../models/human-bigbrain


In [ ]:
# Cell 2 — Upload model weights
#
# Change UPLOAD_SPECIES:
#   "human-bigbrain"  — BigBrain only (default)
#   "mouse"           — Mouse only
#   "human"           — Human Allen only
#   "all"             — All 3 models

from huggingface_hub import HfApi, create_repo

UPLOAD_SPECIES = "human-bigbrain"

if not HF_TOKEN:
    raise RuntimeError("HUGGING_FACE_TOKEN not set.")

api = HfApi(token=HF_TOKEN)
targets = list(MODEL_CONFIGS.keys()) if UPLOAD_SPECIES == "all" else [UPLOAD_SPECIES]

for species in targets:
    cfg = MODEL_CONFIGS[species]
    local_dir = cfg["local_dir"]
    repo_id = f"{HF_USERNAME}/{MODEL_BASE}-{cfg['repo_suffix']}"

    print(f"\n--- {species} -> {repo_id} ---")

    if not local_dir.exists():
        print(f"  SKIP: {local_dir} not found")
        continue

    create_repo(repo_id=repo_id, token=HF_TOKEN, exist_ok=True)

    files = [
        f for f in local_dir.iterdir()
        if f.is_file()
        and f.name not in {"optimizer.pt", "scheduler.pt"}
        and not f.name.startswith("rng_state")
    ]

    for filepath in files:
        mb = filepath.stat().st_size / 1e6
        print(f"  Uploading {filepath.name} ({mb:.1f} MB)...")
        api.upload_file(
            path_or_fileobj=str(filepath),
            path_in_repo=filepath.name,
            repo_id=repo_id,
            token=HF_TOKEN,
        )

    print(f"  Done: https://huggingface.co/{repo_id}")

print("\nModel upload complete.")
print("Next: run upload_papers_to_hf.ipynb locally to add READMEs and papers.")


--- human-bigbrain -> Noel-Niko/dinov2-upernet-20260322-histology-annotation-human-bigbrain ---

/Users/xnxn040/PycharmProjects/Personal-Projects/histological-image-analysis/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  Uploading model.safetensors (1364.5 MB)...


Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  49%|████▊     |  663MB / 1.36GB,  553MB/s  
Processing Files (0 / 1):  49%|████▊     |  665MB / 1.36GB,  475MB/s  
Processing Files (0 / 1):  49%|████▉     |  668MB / 1.36GB,  417MB/s  
Processing Files (0 / 1):  49%|████▉     |  672MB / 1.36GB,  373MB/s  
Processing Files (0 / 1):  49%|████▉     |  674MB / 1.36GB,  337MB/s  
Processing Files (0 / 1):  50%|█████     |  682MB / 1.36GB,  310MB/s  
Processing Files (0 / 1):  50%|█████     |  687MB / 1.36GB,  286MB/s  
Processing Files (0 / 1):  51%|█████     |  691MB / 1.36GB,  266MB/s  
Processing Files (0 / 1):  51%|█████     |  699MB / 1.36GB,  250MB/s  
Processing Files (0 / 1):  51%|█████▏    |  701MB / 1.36GB,  234MB/s  
Processing Files (0 / 1):  52%|█████▏    |  706MB / 1.36GB,  221MB/s  
Processing Files (0 / 1):  52%|█████▏    |  712MB / 1.36GB,  209MB/s  
Processing Files (0 / 1):  52%|█████▏    |  715MB / 1.36GB,  198MB/s  
Processing